In [0]:
%pip install xgboost==3.0.5

# Reinicia o Python para o pacote ficar visível

dbutils.library.restartPython()

In [0]:
import mlflow

MODEL_NAME = "workspace.default.hackathon_databricks_model"

# Listar versões disponíveis
client = mlflow.MlflowClient(registry_uri="databricks-uc")
versions = client.search_model_versions(filter_string=f"name='{MODEL_NAME}'")
for v in sorted(versions, key=lambda x: int(x.version)):
    print(f"v{v.version}: status={v.status}")

# Carregar a versão mais recente (v3)
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/3")
print("\nModelo carregado com sucesso:", MODEL_NAME, "v3")

# Esquema de entrada esperado pelo modelo
print("\nInput schema:")
print(model.metadata.get_input_schema())

In [0]:
# EXECUTAR APENAS UMA VEZ 

import json
from pyspark.sql import functions as F

# 1. Rótulos: nome -> descrição (as descrições orientam o LLM; baseadas nas regras definidas)
labels = {
    "satisfacao": "The customer expresses satisfaction with the service and does NOT mention deciding to cancel, churn or switch to another provider.",
    "preco_alto": "Primary reason is the monthly charges being too high or expensive, without mentioning having found a competitor's better offer.",
    "oferta_concorrente": "Primary reason is having found a better deal, better price or promotional offer from another provider, deciding to switch.",
    "qualidade_servico": "Primary reason is problems with the service itself: slow speeds, unreliable connection, frequent outages or interruptions.",
    "mudanca_endereco": "Primary reason is moving to a new location or address (relocation).",
    "metodo_pagamento": "Primary reason is difficulties or inconvenience with the payment method or billing process (not the price itself).",
    "motivo_pessoal": "Primary reason is personal circumstances, unforeseen situations or financial constraints, unrelated to the provider.",
    "outro": "Any other primary reason not covered by the labels above."
}

# 2. Instruções globais: codificam as regras de classificação
instructions = (
    "Inputs are English customer feedback texts from internet/phone service customers. "
    "Some of them canceled the service or switched providers. "
    "If the customer expresses satisfaction and does not mention canceling or switching, classify as 'satisfacao'. "
    "If the text indicates the customer decided to cancel or switch (even when otherwise satisfied), "
    "classify by the PRIMARY reason stated for that decision. "
    "If the customer expresses dissatisfaction without canceling, classify the PRIMARY reason of the dissatisfaction. "
    "When the customer mentions both high prices and having found a better deal elsewhere, prefer 'oferta_concorrente'."
)

labels_str = json.dumps(labels).replace("'", "''")
instructions_str = instructions.replace("'", "''")

# 3. Classificação: uma chamada de LLM por linha, paralelizada pelo Spark
df_classificado = (
    spark.table("workspace.default.inference")
    .withColumn(
        "motivo_feedback",
        F.expr(f"""
            ai_classify(
                CustomerFeedback,
                '{labels_str}',
                MAP('version', '2.1', 'instructions', '{instructions_str}')
            ):response[0].value::STRING
        """)
    )
)

# 4. Persistir para não pagar a classificação novamente a cada execução
df_classificado.write.mode("overwrite").saveAsTable("workspace.default.inference_classificado")

In [0]:
# Carregar a tabela workspace.default.train_classificado em um DataFrame Pandas
df = spark.table("workspace.default.inference_classificado").toPandas()

print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {df.shape[1]}")

# Amostra dos dados
df.head()


In [0]:
# ============================================================
# FILTRAGEM DE OBSERVACOES
# ============================================================

def filtrar_observacoes_invalidas(df):
    """
    Exclui observacoes onde simultaneamente:

        tenure == 'unknown'
        AND
        TotalCharges == NULL

    Essa regra nao depende de estatisticas da base e, portanto, pode ser executada antes do split.
    Retorna um novo DataFrame sem modificar o original.
    """

    df = df.copy()

    tenure_unknown = (df["tenure"].astype("string").str.strip().str.lower().eq("unknown"))

    total_charges_null = (df["TotalCharges"].isna())

    mascara_excluir = (tenure_unknown & total_charges_null)

    return (df.loc[~mascara_excluir].copy().reset_index(drop=True))


In [0]:
df = filtrar_observacoes_invalidas(df)
df

In [0]:
# ============================================================
# PREDICAO DE CHURN COM O MODELO REGISTRADO (v3)
# ============================================================
import numpy as np
import pandas as pd

THRESHOLD = 0.6

# 1. Montar a entrada a partir do input schema do modelo (garante ordem e tipos esperados)
schema = model.metadata.get_input_schema()
try:
    input_cols = list(schema.input_names())
except Exception:
    input_cols = []
if not input_cols:
    # Fallback: usar todas as colunas disponiveis
    input_cols = list(df.columns)

X = df[input_cols].copy()
print(f"Colunas usadas na predicao: {input_cols}")

# 2. Obter as probabilidades de churn do modelo
raw = model.predict(X)

# Normalizar a saida para um vetor de probabilidades de Churn
if isinstance(raw, pd.DataFrame):
    prob_churn = raw.iloc[:, -1].to_numpy(dtype=float)
else:
    arr = np.asarray(raw)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        # Saida no formato [P(No), P(Yes)] -> usar P(Yes)
        prob_churn = arr[:, 1].astype(float)
    else:
        prob_churn = arr.reshape(-1).astype(float)

# 3. Montar o DataFrame final com probChurn e a classificacao
resultados = df.copy()
resultados["probChurn"] = prob_churn
resultados["churnPredicao"] = np.where(resultados["probChurn"] >= THRESHOLD, "Yes", "No")

print("\nAmostra das predicoes:")
print(resultados[["probChurn", "churnPredicao"]].head())

print("\nDistribuicao das predicoes:")
print(resultados["churnPredicao"].value_counts(normalize=True).rename("proporcao").to_frame())

# 4. Salvar a nova tabela no Unity Catalog
spark.createDataFrame(resultados).write.mode("overwrite").saveAsTable(
    "workspace.default.inference_classificado_predicao"
)

print(f"\nTabela 'workspace.default.inference_classificado_predicao' criada com {len(resultados)} linhas.")

In [0]:
resultados

In [0]:
# ============================================================
# EXPORTAR PREDICOES EM CSV (FORMATO: COLUNAS customerID E prediction)
# ============================================================
import pandas as pd
import datetime

# Uma coluna "customerID" com o valor do customerID
# e uma coluna "prediction" com o valor de churnPredicao
prediction = resultados[["customerID", "churnPredicao"]].rename(
    columns={"churnPredicao": "prediction"}
)

# Salvar no Workspace (pode ser baixado pela interface do Databricks)
#csv_path = "/Workspace/churn_predicoes.csv"
# prediction_Grupo6_20261107_133226
#prediction.to_csv(csv_path, index=False, header=False)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
prediction.to_csv(f'prediction_Grupo6_{timestamp}.csv', index=True, header=True)

print(f"Dimensoes: {prediction.shape[0]} linhas x {prediction.shape[1]} colunas")
print("\nPrevia:")
print(prediction.iloc[:, :6])
